# 0-order optimization

In [ ]:
# pip install -e .
# python -m cell_simulator.search_example

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from cell_simulator.dynamics import HarmonicOscillatorContraction
from cell_simulator.forces import ForceModel
from cell_simulator.geometry import Cell, PillarGrid
from cell_simulator.integrators import RK4
from cell_simulator.observations import TrackingObservationModel
from cell_simulator.simulator import Experiment, ExperimentConfig
from cell_simulator.visualize import *

### Setup physics

In [ ]:
set_style()
grid = PillarGrid.square_lattice(15, 15, 4000.0)
cell = Cell(center_nm=np.array([28900.0, 27400.0]), radius_nm=20000.0)
force_model = ForceModel(F_max=600.0, k=4.8)
times = np.arange(0.0, 30.0 + 1e-9, 0.5)

### Setup simulation + loss

In [ ]:
def simulate(omega, phase, observation_model=None):
    cfg = ExperimentConfig(
        pillar_grid=grid,
        cell=cell,
        dynamics=HarmonicOscillatorContraction(omega, phase),
        integrator=RK4(),
        force_model=force_model,
        observation_model=observation_model,
    )
    return Experiment(cfg).run(times, steps_per_frame=10)


def loss(result):
    return float(np.mean((result.forces - target) ** 2))

### Define GT parameters

In [ ]:
true_omega, true_phase = 2 * np.pi / 10.0, 0.8
true_data = simulate(true_omega, true_phase, TrackingObservationModel(2.0, rng=np.random.default_rng(1)))
target = true_data.observed_forces

noise_floor = loss(true_data)  # true parameters, noise-free prediction vs noisy target
print(f"Noise floor: {noise_floor:.2f} i.e. minimum achievable loss with perfect parameters")

### Simulate random parameters

In [ ]:
N = 1000
rng = np.random.default_rng(0)

# Sample random parameters
omegas = np.abs(rng.normal(0.6, 0.15, N))
phis = rng.normal(0.0, 1.5, N)

# Simulate
candidates = [simulate(w, p) for w, p in zip(omegas, phis)]

# Score candidates
losses = np.array([loss(r) for r in candidates])
best, worst = candidates[int(np.argmin(losses))], candidates[int(np.argmax(losses))]

In [ ]:
fig = plot_parameter_search(omegas, phis, losses, true_omega, true_phase, noise_floor)

(a) The yellow region runs diagonally: a slightly higher ω pairs with a slightly lower φ, and vice versa. The two parameters partly compensate for each other. A too fast frequency's loss can be offset by a shifted phase starting behind such that it does not "run off" too much within our interval. 

(b) Largest loss is at true $\omega$ (interesting) but at the opposite phase, shifted such that it is always opposite. Away from the true $\omega$ the loss plateaus around $8\cdot 10^3$ as the frequency is very wrong and the phase is less important. 

(c) Many true $\phi$ have high loss because their $\omega$ is so wrong. 

The plots also underlines that random sampling is a wasteful tactic. Only a handful og samples lands inside the area of interest (a) i.e. funnels (b, c) and only by luck. 

### As grid search (takes 3 min)

In [ ]:
omega_grid = np.linspace(0.2, 1.1, 100)
phi_grid = np.linspace(-np.pi, np.pi, 100)
loss_grid = np.array([[loss(simulate(w, p)) for w in omega_grid] for p in phi_grid])
fig, ax = plt.subplots(figsize=(6.5, 4.8))
fig = plot_loss_landscape(omega_grid, phi_grid, loss_grid, true_omega, true_phase, samples=(omegas, phis), ax=ax)

### Compare worst, best and true fit

In [ ]:
edge_pillar = int(np.argmax(true_data.rho * true_data.under_cell))  # strongest-force pillar
fig = plot_fit_comparison(true_data, best, worst, pillar_idx=edge_pillar)

Again notice, true $\omega$ opposite $\phi$ is the worst fit. Also notice that the best fit accumulate the loss as time progresses as the frequency is a bit (0.002) slower than the true value. 

In [ ]:
from IPython.display import HTML

anim = animate_fit_comparison(true_data, best, worst, fps=5)
plt.close(anim._fig)  # avoid an extra static copy below the video
HTML(anim.to_jshtml())

In [ ]:
from IPython.display import HTML

anim = animate_pillar_motion(true_data, edge_pillar, fps=5)
plt.close(anim._fig)  # avoid an extra static copy below the video
HTML(anim.to_jshtml())

Different values for $u_x$ and $u_y$ because the pillar (175) is not placed with the same distance from the cell center so the force vector is different lengths in the two dimensions.